### Reclamações

In [2]:
# 1. Importar bibliotecas
import os
import pandas as pd
import sqlite3

# 2. Garantir que a pasta data existe
os.makedirs("../data", exist_ok=True)

# 3. Criar (ou abrir) o banco de dados
db_path = "../data/nubank.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Banco criado em: {db_path}")

# 4. Ler todos os CSVs da pasta processed e salvar como tabelas
processed_path = "../data/processed"

for file in os.listdir(processed_path):
    if file.endswith(".csv"):
        table_name = file.replace(".csv", "")
        df = pd.read_csv(os.path.join(processed_path, file), sep=",", encoding="utf-8-sig")
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Tabela {table_name} criada com {len(df)} registros.")

Banco criado em: ../data/nubank.db
Tabela dados_nubank_extraidos criada com 125 registros.
Tabela reclamacoes_com_clusters criada com 122 registros.
Tabela reclamacoes_com_insights criada com 122 registros.


In [4]:
# 5. Listar todas as tabelas criadas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [t[0] for t in cursor.fetchall()]
print("Tabelas criadas:", tables)

Tabelas criadas: ['dados_nubank_extraidos', 'reclamacoes_com_clusters', 'reclamacoes_com_insights']


In [5]:
# 6. Mostrar estrutura de cada tabela
for table in tables:
    cursor.execute(f"PRAGMA table_info({table});")
    print(f"\nEstrutura da tabela {table}:")
    for col in cursor.fetchall():
        print(col)


Estrutura da tabela dados_nubank_extraidos:
(0, 'Reclamacao', 'TEXT', 0, None, 0)

Estrutura da tabela reclamacoes_com_clusters:
(0, 'Reclamacao', 'TEXT', 0, None, 0)
(1, 'Categoria', 'TEXT', 0, None, 0)
(2, 'Cluster', 'INTEGER', 0, None, 0)
(3, 'Comprimento', 'INTEGER', 0, None, 0)

Estrutura da tabela reclamacoes_com_insights:
(0, 'Reclamacao', 'TEXT', 0, None, 0)
(1, 'Categoria_Primaria', 'TEXT', 0, None, 0)
(2, 'Severidade_Estimada', 'TEXT', 0, None, 0)
(3, 'Urgencia', 'INTEGER', 0, None, 0)
(4, 'Perda_Financeira', 'INTEGER', 0, None, 0)
(5, 'Problema_Funcional', 'INTEGER', 0, None, 0)
(6, 'Problema_Seguranca', 'INTEGER', 0, None, 0)
(7, 'Cluster', 'INTEGER', 0, None, 0)


In [6]:
# 7. Fazer uma consulta simples em uma tabela
cursor.execute("SELECT * FROM dados_nubank_extraidos LIMIT 5;")
print(cursor.fetchall())

[('Nubank falha ao contestar transação via PIX e cliente perde R$ 200,00 em [Editado pelo Reclame Aqui]',), ('249,93',), ('Bloqueio e Desativação de Conta Nubank sem Permissão ou Motivo',), ('Banco não libera crédito nem aumenta limite mesmo com pagamento em dia e movimentação alta',), ('Não consigo acessar minha conta após troca de aparelho celular, impactando pagamentos.',)]


In [7]:
# 8. Fechar a conexão quando terminar
conn.close()

### Selic

In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

# Caminho do arquivo CSV
csv_path = Path.cwd().parent / "data" / "raw" / "bcdata.sgs.11.csv"

# Ler o CSV
df_selic = pd.read_csv(csv_path, sep=";", decimal=",")
df_selic["data"] = pd.to_datetime(df_selic["data"], dayfirst=True)
df_selic.rename(columns={"valor": "selic"}, inplace=True)

# Conectar ao banco SQLite
db_path = Path.cwd().parent / "data" / "nubank.db"
conn = sqlite3.connect(db_path)

# Salvar como tabela no banco
df_selic.to_sql("selic_diaria", conn, if_exists="replace", index=False)

# Fechar conexão
conn.close()